# S4 - ML Distribuido con Spark MLlib (Regresión)
## Proyecto Sello — Contaminación del Agua

Actividad individual (sección 4 de la guía S4), aplicada a `lecturas_agua.csv`
(sensores de agua en Río Coata, Juliaca y Cabanillas).

**Nota sobre integración de fuentes:** el caso de este proyecto no tiene múltiples
orígenes de datos separados, es una sola fuente ya integrada por diseño, tal como
permite explícitamente 4.1: *"preparar una sola fuente ya integrada, si el caso del
equipo no tiene múltiples orígenes"*. Toda la calidad de datos (esquema, nulos,
duplicados) se aplica igual sobre esa única fuente.

## Fase 1 — Business Understanding

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion4-ml-distribuido-regresion-agua")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/10 01:59:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
ORIGEN_DATOS = "/opt/s04-ml-distribuido-regresion/data"

**Objetivo:** estimar `plomo_mg_l` (concentración de plomo en el agua) a partir de
variables físico-químicas medidas en el mismo instante por sensores continuos
(pH, turbidez, temperatura, conductividad, sólidos disueltos, oxígeno disuelto y
caudal) — no un pronóstico con historial temporal (eso queda para S10).

**Por qué esta variable:** el análisis de metales pesados en laboratorio (plomo,
arsénico, mercurio, cadmio) es más lento y costoso que los sensores físico-químicos
continuos. Si un modelo puede estimar `plomo_mg_l` razonablemente bien a partir de
sensores baratos y en tiempo real, eso habilita una alerta temprana de contaminación
por plomo sin esperar el resultado de laboratorio.

**Alcance:** comparar un modelo base de regresión lineal, tres configuraciones de
regularización y un segundo algoritmo (Random Forest), reportando RMSE, R² y MAE — sin
búsqueda exhaustiva de hiperparámetros. El modelo ganador decide si un sistema de
alerta temprana basado solo en sensores físico-químicos es viable, o si de verdad se
necesita seguir dependiendo del análisis de laboratorio para detectar plomo.

## Fase 2 — Data Understanding

Fuente única ya integrada. Se carga con esquema
explícito — el mismo de S3, reutilizado — y se explora antes de decidir ninguna regla
de limpieza.

In [3]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
)

schema_agua = StructType([
    StructField("id_lectura", IntegerType(), True),
    StructField("sensor_id", StringType(), True),
    StructField("ubicacion", StringType(), True),
    StructField("fecha_hora", TimestampType(), True),
    StructField("canal_transmision_id", IntegerType(), True),
    StructField("ph", DoubleType(), True),
    StructField("turbidez_ntu", DoubleType(), True),
    StructField("temperatura_c", DoubleType(), True),
    StructField("conductividad_us_cm", DoubleType(), True),
    StructField("solidos_disueltos_totales_mg_l", DoubleType(), True),
    StructField("oxigeno_disuelto_mg_l", DoubleType(), True),
    StructField("plomo_mg_l", DoubleType(), True),
    StructField("arsenico_mg_l", DoubleType(), True),
    StructField("mercurio_mg_l", DoubleType(), True),
    StructField("cadmio_mg_l", DoubleType(), True),
    StructField("coliformes_fecales_nmp_100ml", DoubleType(), True),
    StructField("escherichia_coli_nmp_100ml", DoubleType(), True),
    StructField("presencia_parasitos", IntegerType(), True),
    StructField("radiactividad_bq_l", DoubleType(), True),
    StructField("caudal_l_s", DoubleType(), True),
    StructField("indice_riesgo_normalizado", DoubleType(), True),
])

df_agua = spark.read.csv(
    f"{ORIGEN_DATOS}/lecturas_agua.csv",
    header=True,
    schema=schema_agua,
)

df_agua.printSchema()
print(f"Filas: {df_agua.count():,}")

root
 |-- id_lectura: integer (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- ubicacion: string (nullable = true)
 |-- fecha_hora: timestamp (nullable = true)
 |-- canal_transmision_id: integer (nullable = true)
 |-- ph: double (nullable = true)
 |-- turbidez_ntu: double (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- conductividad_us_cm: double (nullable = true)
 |-- solidos_disueltos_totales_mg_l: double (nullable = true)
 |-- oxigeno_disuelto_mg_l: double (nullable = true)
 |-- plomo_mg_l: double (nullable = true)
 |-- arsenico_mg_l: double (nullable = true)
 |-- mercurio_mg_l: double (nullable = true)
 |-- cadmio_mg_l: double (nullable = true)
 |-- coliformes_fecales_nmp_100ml: double (nullable = true)
 |-- escherichia_coli_nmp_100ml: double (nullable = true)
 |-- presencia_parasitos: integer (nullable = true)
 |-- radiactividad_bq_l: double (nullable = true)
 |-- caudal_l_s: double (nullable = true)
 |-- indice_riesgo_normalizado: double (nullab

[Stage 0:===>                                                     (1 + 15) / 16]

Filas: 1,500,000


Conteo de nulos por columna (Data Understanding, todavía sin tratar nada):

In [4]:
from pyspark.sql.functions import col, count as spark_count, when

df_agua.select([
    spark_count(when(col(c).isNull(), c)).alias(c) for c in df_agua.columns
]).show(vertical=True, truncate=False)


[Stage 3:===>                                                     (1 + 15) / 16]

-RECORD 0-------------------------------
 id_lectura                     | 0     
 sensor_id                      | 0     
 ubicacion                      | 0     
 fecha_hora                     | 0     
 canal_transmision_id           | 0     
 ph                             | 0     
 turbidez_ntu                   | 0     
 temperatura_c                  | 0     
 conductividad_us_cm            | 0     
 solidos_disueltos_totales_mg_l | 0     
 oxigeno_disuelto_mg_l          | 60000 
 plomo_mg_l                     | 0     
 arsenico_mg_l                  | 0     
 mercurio_mg_l                  | 0     
 cadmio_mg_l                    | 0     
 coliformes_fecales_nmp_100ml   | 60000 
 escherichia_coli_nmp_100ml     | 0     
 presencia_parasitos            | 0     
 radiactividad_bq_l             | 0     
 caudal_l_s                     | 0     
 indice_riesgo_normalizado      | 0     



Estadísticas descriptivas de las variables físico-químicas, sin limpiar todavía:

In [5]:
VARIABLES_FISICOQUIMICAS = [
    "ph", "turbidez_ntu", "temperatura_c", "conductividad_us_cm",
    "solidos_disueltos_totales_mg_l", "oxigeno_disuelto_mg_l", "caudal_l_s",
]

df_agua.describe(VARIABLES_FISICOQUIMICAS + ["plomo_mg_l"]).show()


26/09/10 02:08:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 6:===>                                                     (1 + 15) / 16]

+-------+------------------+------------------+------------------+-------------------+------------------------------+---------------------+-----------------+--------------------+
|summary|                ph|      turbidez_ntu|     temperatura_c|conductividad_us_cm|solidos_disueltos_totales_mg_l|oxigeno_disuelto_mg_l|       caudal_l_s|          plomo_mg_l|
+-------+------------------+------------------+------------------+-------------------+------------------------------+---------------------+-----------------+--------------------+
|  count|           1500000|           1500000|           1500000|            1500000|                       1500000|              1440000|          1500000|             1500000|
|   mean| 6.779985460000028|23.894756653333424|10.436569366666657|  575.9822721333334|             364.8078741333334|     6.03224152777778|79.87301719999984|0.030705033733334528|
| stddev|0.5663791440182202| 21.34430490748802| 2.928422767849878|  334.3747691787466|            221.314

Correlación de cada variable física con la variable objetivo (`plomo_mg_l`) — una primera lectura, todavía sobre datos sin limpiar:

In [6]:
for columna in VARIABLES_FISICOQUIMICAS:
    correlacion = df_agua.stat.corr(columna, "plomo_mg_l")
    print(f"{columna:32s} correlacion con plomo_mg_l: {correlacion:.4f}")


ph                               correlacion con plomo_mg_l: -0.5297


turbidez_ntu                     correlacion con plomo_mg_l: 0.5457


temperatura_c                    correlacion con plomo_mg_l: 0.3049


conductividad_us_cm              correlacion con plomo_mg_l: 0.6230


solidos_disueltos_totales_mg_l   correlacion con plomo_mg_l: 0.5960


oxigeno_disuelto_mg_l            correlacion con plomo_mg_l: -0.3984


[Stage 27:===>                                                    (1 + 15) / 16]

caudal_l_s                       correlacion con plomo_mg_l: 0.6774


> Referencia (sobre una muestra): `caudal_l_s` ≈ 0.68, `conductividad_us_cm` ≈ 0.63,
`solidos_disueltos_totales_mg_l` ≈ 0.60, `turbidez_ntu` ≈ 0.55, `ph` ≈ -0.53,
`oxigeno_disuelto_mg_l` ≈ -0.50, `temperatura_c` ≈ 0.31. Tus valores exactos sobre el
dataset completo pueden variar levemente.

## Fase 3 — Data Preparation

Con los datos ya conocidos (Fase 2), acá se aplican las reglas de limpieza: casos
especiales (valores fuera de dominio), duplicados y nulos — antes de ensamblar el
vector de predictores.

**3.1 Casos especiales — validación de dominio.** Ningún sensor de este dataset usa un código de error tipo `99999`, pero igual se confirma que los valores están dentro de un rango físicamente posible, antes de asumirlo:

In [7]:
anomalias_ph = df_agua.filter((col("ph") < 0) | (col("ph") > 14)).count()
anomalias_turbidez = df_agua.filter(col("turbidez_ntu") < 0).count()
anomalias_caudal = df_agua.filter(col("caudal_l_s") < 0).count()

print(f"pH fuera de [0,14]: {anomalias_ph}")
print(f"turbidez negativa: {anomalias_turbidez}")
print(f"caudal negativo: {anomalias_caudal}")

[Stage 36:===>                                                    (1 + 15) / 16]

pH fuera de [0,14]: 0
turbidez negativa: 0
caudal negativo: 0


Si los tres conteos dan 0, es un control de calidad exitoso — no un resultado vacío sin valor: confirma que no hay valores fuera de dominio que limpiar en estas columnas.

**3.2 Duplicados.** Diagnóstico primero, sin eliminar nada — ¿la misma lectura de un sensor llegó más de una vez, con la misma marca de tiempo?

In [8]:
df_agua.groupBy("sensor_id", "fecha_hora").count().filter("count > 1").show(10)

total_grupos_duplicados = (
    df_agua.groupBy("sensor_id", "fecha_hora").count().filter("count > 1").count()
)
print(f"Combinaciones sensor_id+fecha_hora con mas de una lectura: {total_grupos_duplicados:,}")


+---------+-------------------+-----+
|sensor_id|         fecha_hora|count|
+---------+-------------------+-----+
|     0156|2025-06-11 09:00:00|    3|
|     0119|2025-04-17 04:00:00|    2|
|     0230|2025-04-10 00:00:00|    2|
|     0185|2024-03-01 14:00:00|    2|
|     0195|2024-07-04 04:00:00|    2|
|     0094|2024-01-07 04:00:00|    2|
|     0193|2024-10-29 15:00:00|    2|
|     0114|2024-06-26 15:00:00|    2|
|     0068|2025-09-12 09:00:00|    2|
|     0140|2025-08-26 07:00:00|    2|
+---------+-------------------+-----+
only showing top 10 rows


[Stage 42:===>                                                    (1 + 15) / 16]

Combinaciones sensor_id+fecha_hora con mas de una lectura: 169,995


Se resuelve con `Window + row_number()` entre dos lecturas del mismo sensor en el mismo instante,
se conserva la que tiene menos nulos, no una cualquiera:

In [9]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, lit

columnas_con_nulos_posibles = ["oxigeno_disuelto_mg_l", "coliformes_fecales_nmp_100ml"]

df_con_conteo_nulos = df_agua.withColumn(
    "CantidadNulos",
    sum(when(col(c).isNull(), 1).otherwise(0) for c in columnas_con_nulos_posibles),
)

ventana_dedup = Window.partitionBy("sensor_id", "fecha_hora").orderBy(
    col("CantidadNulos").asc(),
    col("id_lectura").asc(),   # desempate determinista: id_lectura es unico por fila
)

df_sin_duplicados = (
    df_con_conteo_nulos
    .withColumn("row_num", row_number().over(ventana_dedup))
    .filter(col("row_num") == 1)
    .drop("row_num", "CantidadNulos")
)

total_antes = df_agua.count()
total_despues = df_sin_duplicados.count()
print(f"Filas antes: {total_antes:,}, despues de deduplicar: {total_despues:,}")
print(f"Filas eliminadas por duplicado sensor_id+fecha_hora: {total_antes - total_despues:,}")

[Stage 51:===>                                                    (1 + 15) / 16]

Filas antes: 1,500,000, despues de deduplicar: 1,313,822
Filas eliminadas por duplicado sensor_id+fecha_hora: 186,178


In [10]:
# Confirmacion: ya no deberia quedar ninguna combinacion sensor_id+fecha_hora repetida
sin_duplicar_verificacion = df_sin_duplicados.dropDuplicates(["sensor_id", "fecha_hora"]).count()
assert df_sin_duplicados.count() == sin_duplicar_verificacion, "Todavia hay sensor_id+fecha_hora duplicado"
print("Verificado: no queda ninguna combinacion sensor_id+fecha_hora duplicada.")

[Stage 63:===>                                                    (1 + 15) / 16]

Verificado: no queda ninguna combinacion sensor_id+fecha_hora duplicada.


**3.3 Nulos.** Con los duplicados resueltos, se eliminan solo las filas que quedan sin dato en las columnas que el modelo necesita (predictores + objetivo) — no toda fila con cualquier nulo en cualquier columna:

In [11]:
PREDICTORES = [
    "ph", "turbidez_ntu", "temperatura_c", "conductividad_us_cm",
    "solidos_disueltos_totales_mg_l", "oxigeno_disuelto_mg_l", "caudal_l_s",
]
OBJETIVO = "plomo_mg_l"

antes = df_sin_duplicados.count()
df_valido = df_sin_duplicados.na.drop(subset=PREDICTORES + [OBJETIVO])
despues = df_valido.count()

print(f"Filas antes: {antes:,}")
print(f"Filas despues de na.drop(subset=PREDICTORES+[OBJETIVO]): {despues:,}")
print(f"Filas eliminadas por nulos en columnas criticas para el modelo: {antes - despues:,}")

assert df_valido.filter(col(OBJETIVO).isNull()).count() == 0

df_valido = df_valido.cache()


Filas antes: 1,313,822
Filas despues de na.drop(subset=PREDICTORES+[OBJETIVO]): 1,267,521
Filas eliminadas por nulos en columnas criticas para el modelo: 46,301


> Con esto, `df_valido` es la capa **Silver**: esquema validado (2.2), casos especiales
revisados, duplicados resueltos con `Window+row_number()`, y nulos tratados solo sobre
las columnas críticas para el modelo.

**3.4 Escritura particionada (Silver → Gold).** Se reutiliza `ubicacion` como columna de partición — la misma que en S3: solo 4 valores distintos, y es la columna por la que se filtra en el análisis del proyecto. No hace falta derivar una columna de fecha para particionar, porque ya existe una columna categórica de baja cardinalidad en los datos (a diferencia del ejemplo del docente, que no la tenía):

In [12]:
ARTIFACTS = "/opt/s04-ml-distribuido-regresion/artifacts"

(
    df_valido
    .repartition(4)
    .write.format("parquet")
    .mode("overwrite")
    .partitionBy("ubicacion")
    .save(f"{ARTIFACTS}/lecturas_agua_gold")
)

import os
for carpeta in sorted(os.listdir(f"{ARTIFACTS}/lecturas_agua_gold")):
    print(carpeta)


._SUCCESS.crc
_SUCCESS
ubicacion=Cabanillas (ciudad)
ubicacion=Cabanillas (nacimiento de agua)
ubicacion=Juliaca (urbano)
ubicacion=Rio Coata


Lectura de vuelta y verificación — igual que en S3, no se asume, se confirma:

In [13]:
df_verificacion = spark.read.parquet(f"{ARTIFACTS}/lecturas_agua_gold")
df_verificacion.printSchema()

assert df_verificacion.count() == df_valido.count()
print(f"Verificado: {df_verificacion.count():,} filas, ida y vuelta sin perdida.")

df_verificacion.filter(col("ubicacion") == "Rio Coata").explain(True)


root
 |-- id_lectura: integer (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- fecha_hora: timestamp (nullable = true)
 |-- canal_transmision_id: integer (nullable = true)
 |-- ph: double (nullable = true)
 |-- turbidez_ntu: double (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- conductividad_us_cm: double (nullable = true)
 |-- solidos_disueltos_totales_mg_l: double (nullable = true)
 |-- oxigeno_disuelto_mg_l: double (nullable = true)
 |-- plomo_mg_l: double (nullable = true)
 |-- arsenico_mg_l: double (nullable = true)
 |-- mercurio_mg_l: double (nullable = true)
 |-- cadmio_mg_l: double (nullable = true)
 |-- coliformes_fecales_nmp_100ml: double (nullable = true)
 |-- escherichia_coli_nmp_100ml: double (nullable = true)
 |-- presencia_parasitos: integer (nullable = true)
 |-- radiactividad_bq_l: double (nullable = true)
 |-- caudal_l_s: double (nullable = true)
 |-- indice_riesgo_normalizado: double (nullable = true)
 |-- ubicacion: string (nullab

In [14]:
df_verificacion.groupBy("ubicacion").count().orderBy("ubicacion").show(truncate=False)

+-------------------------------+------+
|ubicacion                      |count |
+-------------------------------+------+
|Cabanillas (ciudad)            |306464|
|Cabanillas (nacimiento de agua)|260149|
|Juliaca (urbano)               |350563|
|Rio Coata                      |350345|
+-------------------------------+------+



**Capas de mi pipeline (Proyecto Sello - Agua, sesión S4):**

- **Bronze (raw):** `lecturas_agua.csv` tal como llega de los sensores, sin validar.
- **Silver:** `df_valido` — esquema explícito, casos especiales revisados, duplicados
  resueltos (`sensor_id` + `fecha_hora` con `Window+row_number()`) y nulos tratados
  solo sobre las columnas críticas para el modelo.
- **Gold:** `lecturas_agua_gold/`, particionado por `ubicacion`, la salida que alimenta
  el modelado de esta misma sesión.

In [15]:
df = df_verificacion
df_valido.unpersist()

print(f"Filas para modelar: {df.count():,}")

Filas para modelar: 1,267,521


**3.5 Ensamblar el vector de predictores.**

In [16]:
from pyspark.ml.feature import VectorAssembler

ensamblador = VectorAssembler(inputCols=PREDICTORES, outputCol="features")
df_ml = ensamblador.transform(df).select("features", OBJETIVO)

df_ml.show(5, truncate=False)

+------------------------------------------+----------+
|features                                  |plomo_mg_l|
+------------------------------------------+----------+
|[6.75,45.74,14.68,1137.3,573.5,7.38,213.1]|0.0776    |
|[7.19,48.96,12.11,699.4,576.4,6.01,105.5] |0.1366    |
|[5.45,57.52,7.53,1041.7,211.4,5.03,254.4] |0.093     |
|[5.92,65.74,15.28,756.2,694.9,3.7,241.5]  |0.0606    |
|[6.85,50.91,5.37,1366.2,479.2,2.38,138.9] |0.1529    |
+------------------------------------------+----------+
only showing top 5 rows


**3.6 Dividir en entrenamiento y prueba.**

In [17]:
df_train, df_test = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Entrenamiento: {df_train.count():,} filas")
print(f"Prueba: {df_test.count():,} filas")


Entrenamiento: 1,014,144 filas


[Stage 117:===>                                                   (1 + 15) / 16]

Prueba: 253,377 filas


**3.7 Escalar si aplica.** No aplica en esta sesión: `LinearRegression` de Spark MLlib
tiene `standardization=True` por defecto, así que ya estandariza los predictores
internamente antes de ajustar el modelo (para que `regParam` penalice de forma justa
entre `conductividad_us_cm`, en cientos, y `ph`, en unidades) y devuelve los
coeficientes en la escala original. `RandomForestRegressor` tampoco lo necesita — sus
árboles dividen por umbrales, no por magnitud.

## Fase 4 — Modeling

In [18]:
from pyspark.ml.regression import LinearRegression

lr_base = LinearRegression(featuresCol="features", labelCol=OBJETIVO)
modelo_base = lr_base.fit(df_train)

print("Coeficientes:", modelo_base.coefficients)
print("Intercepto:", modelo_base.intercept)


26/09/10 02:29:19 WARN Instrumentation: [d442de48] regParam is zero, which might cause numerical instability and overfitting.
netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory
                                                                                

Coeficientes: [-0.007296152554623491,0.00013900043504288246,2.469001045162192e-05,1.6858668929748612e-05,1.705198614319902e-05,-0.0007259852837837078,0.000225009638317568]
Intercepto: 0.04710606544047098


Es normal ver el aviso `regParam is zero, which might cause numerical instability and overfitting` — este modelo base se entrena a propósito sin regularización, como línea de comparación.

In [19]:
from pyspark.ml.evaluation import RegressionEvaluator

def evaluar(predicciones, nombre):
    resultados = {}
    for metrica in ["rmse", "r2", "mae"]:
        evaluador = RegressionEvaluator(labelCol=OBJETIVO, predictionCol="prediction", metricName=metrica)
        resultados[metrica.upper()] = evaluador.evaluate(predicciones)
    print(f"{nombre}: RMSE={resultados['RMSE']:.4f}  R2={resultados['R2']:.4f}  MAE={resultados['MAE']:.4f}")
    return resultados

predicciones_base = modelo_base.transform(df_test)
predicciones_base.select(OBJETIVO, "prediction").show(5)

resultados_base = evaluar(predicciones_base, "LinearRegression base")


+----------+-------------------+
|plomo_mg_l|         prediction|
+----------+-------------------+
|     0.116|0.08668378030471455|
|    0.1224|0.08284557025275423|
|    0.0578| 0.0806018686798807|
|    0.1373|0.08627437452051166|
|    0.1201|0.06734659481692282|
+----------+-------------------+
only showing top 5 rows


LinearRegression base: RMSE=0.0290  R2=0.5136  MAE=0.0183


> Referencia (sobre una muestra): RMSE ≈ 0.029, R² ≈ 0.52, MAE ≈ 0.018. valores exactos sobre el dataset completo en Spark pueden variar levemente.

**Comparar configuraciones de regularización:**

In [20]:
configuraciones = [
    {"nombre": "Sin regularizacion", "regParam": 0.0, "elasticNetParam": 0.0},
    {"nombre": "Ridge (L2)", "regParam": 0.1, "elasticNetParam": 0.0},
    {"nombre": "Elastic Net (L1+L2)", "regParam": 0.1, "elasticNetParam": 0.5},
]

comparacion_configs = []
for config in configuraciones:
    lr = LinearRegression(
        featuresCol="features", labelCol=OBJETIVO,
        regParam=config["regParam"], elasticNetParam=config["elasticNetParam"],
    )
    modelo = lr.fit(df_train)
    predicciones = modelo.transform(df_test)
    resultado = evaluar(predicciones, config["nombre"])
    resultado["Configuracion"] = config["nombre"]
    comparacion_configs.append(resultado)

import pandas as pd
pd.DataFrame(comparacion_configs)[["Configuracion", "RMSE", "R2", "MAE"]]


26/09/10 02:31:07 WARN Instrumentation: [acc9ca01] regParam is zero, which might cause numerical instability and overfitting.
                                                                                

Sin regularizacion: RMSE=0.0290  R2=0.5136  MAE=0.0183


Ridge (L2): RMSE=0.0314  R2=0.4278  MAE=0.0204


[Stage 178:=================>                                     (5 + 11) / 16]

Elastic Net (L1+L2): RMSE=0.0416  R2=-0.0000  MAE=0.0310


,Configuracion,RMSE,R2,MAE
0,Sin regularizacion,0.028987,5.136202e-01,0.018293
1,Ridge (L2),0.031440,4.278119e-01,0.020446
2,Elastic Net (L1+L2),0.041564,-9.359177e-08,0.031006


**Entrenar un segundo algoritmo (Random Forest), sin el supuesto de linealidad:**

In [21]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(featuresCol="features", labelCol=OBJETIVO, numTrees=50, maxDepth=8, seed=42)
modelo_rf = rf.fit(df_train)

predicciones_rf = modelo_rf.transform(df_test)
resultados_rf = evaluar(predicciones_rf, "Random Forest")

26/09/10 02:32:18 WARN DAGScheduler: Broadcasting large task binary with size 1051.3 KiB
26/09/10 02:32:20 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
[Stage 207:======>                                                (2 + 14) / 16]

Random Forest: RMSE=0.0272  R2=0.5708  MAE=0.0157


> Referencia (sobre una muestra): RMSE ≈ 0.028, R² ≈ 0.57, MAE ≈ 0.016 — una mejora modesta pero real sobre el modelo lineal.

**Importancia de variables:**

In [22]:
importancias = list(zip(PREDICTORES, modelo_rf.featureImportances.toArray()))
importancias.sort(key=lambda x: x[1], reverse=True)

for variable, importancia in importancias:
    print(f"{variable:32s} {importancia:.4f}")

caudal_l_s                       0.5591
conductividad_us_cm              0.2081
solidos_disueltos_totales_mg_l   0.1242
turbidez_ntu                     0.0573
ph                               0.0437
oxigeno_disuelto_mg_l            0.0071
temperatura_c                    0.0006


> Referencia (sobre una muestra): `caudal_l_s` concentra la mayor parte de la
importancia (~0.83) — tiene sentido físico: un caudal mayor diluye la concentración de
contaminantes, así que el flujo de agua está directamente relacionado con cuánto plomo
por litro se mide. Muy por debajo: `conductividad_us_cm` (~0.10) y el resto de
variables con aportes menores.

## Fase 5 — Evaluation

Comparar los 4 candidatos entrenados en la Fase 4, sobre el mismo `df_test`:

In [23]:
comparacion_final = comparacion_configs + [
    {**resultados_rf, "Configuracion": "Random Forest"}
]

tabla_final = pd.DataFrame(comparacion_final)[["Configuracion", "RMSE", "R2", "MAE"]]
tabla_final

,Configuracion,RMSE,R2,MAE
0,Sin regularizacion,0.028987,5.136202e-01,0.018293
1,Ridge (L2),0.031440,4.278119e-01,0.020446
2,Elastic Net (L1+L2),0.041564,-9.359177e-08,0.031006
3,Random Forest,0.027231,5.707650e-01,0.015721


**Selección del modelo:** con base en la tabla anterior, **Random Forest** queda
seleccionado como modelo ganador — mejora sobre las tres configuraciones lineales en
las tres métricas a la vez (RMSE más bajo, R² más alto, MAE más bajo), aunque el
margen es modesto, no dramático. Entre las configuraciones lineales, agregar
regularización no mejoró el resultado de forma notable — señal de que el modelo base
no estaba sobreajustando de forma importante.

**Validación contra el objetivo de negocio (3.1.3):** un R² de aproximadamente 0.5-0.6
significa que el modelo explica poco más de la mitad de la variabilidad real de
`plomo_mg_l` a partir de solo 7 variables físico-químicas — suficiente para una alerta
temprana de riesgo relativo (por ejemplo, marcar lecturas con probabilidad alta de
contaminación por plomo para revisión prioritaria), pero no suficiente para reemplazar
por completo el análisis de laboratorio en decisiones críticas de salud pública. El
alcance definido en 3.1.3 —comparar cuatro configuraciones y reportar las tres
métricas— se cumplió.

## Fase 6 — Deployment

In [24]:
modelo_ganador = modelo_rf

modelo_ganador.write().overwrite().save(f"{ARTIFACTS}/modelo_plomo_regresion")
print(f"Modelo guardado en {ARTIFACTS}/modelo_plomo_regresion")

Modelo guardado en /opt/s04-ml-distribuido-regresion/artifacts/modelo_plomo_regresion


## Cierre — Documentar hallazgos y reflexión

**Reflexión técnica (5-8 líneas):** ¿por qué confiar en un único modelo, sin comparar
configuraciones ni algoritmos alternativos, es una decisión de riesgo cuando ese
modelo va a usarse para tomar decisiones reales?

_(Según los resultados del notebook, confiar en un único modelo sería una decisión de riesgo porque no todos los modelos tienen el mismo desempeño ni el mismo nivel de error. Al comparar cuatro alternativas, se observó que Random Forest obtuvo los mejores resultados, superando a las tres configuraciones lineales evaluadas, si solo se hubiera utilizado el modelo lineal sin regularización, se habría elegido una solución menos precisa, además de eso un modelo incorrecto podría generar falsos negativos o falsos positivos, comparar varios modelos permite identificar cuál generaliza mejor, reduce los errores y ofrece mayor confianza para apoyar la toma de decisiones.)_